<div class="alert alert-warning" style="padding: 15px; border: 1px solid #ffeeba; border-radius: 4px; background-color: #fff3cd; color: #856404; margin-bottom: 20px;">
    <strong>⚠️ Warning: Experimental Notebook</strong><br>
    This notebook is experimental and is not guaranteed to work on all workspaces. It requires specific EOxHub workspace setup configurations (such as active PostGIS database credentials, a mounted public bucket folder, and STAC/dynamic-vector-tile endpoints).
</div>

# eoAPI Timeseries Ingestion and Registration Example

This notebook provides a minimal, complete example demonstrating how to ingest, split, view, and register a timeseries dataset in `eoAPI`.

### The Challenges with Spatiotemporal Timeseries Data:
- In many timeseries datasets, the **spatial geometries** (polygons or points of counties, sensors, or regions) stay exactly the same, but the **timeseries records** (numeric readings, weather parameters, or counts) change over hundreds or thousands of steps.
- Duplicating the geometry for every step leads to large, inefficient databases and massive vector tiles.
- **The Solution:** We ingest geometries and timeseries into **separate database tables** and join them dynamically on demand.

### Overview of Steps:
1. **Separation of Geometries & Timeseries:** Ingest a clean, normalized dataset where geometries reside in one table and timeseries data in another.
2. **PostGIS view-like Functions:** Write PL/pgSQL database functions that serve as custom collection layers in TiPg:
   - A date-filtered view function: returns joined geometries and timeseries for a given date parameter.
   - An ID-filtered timeseries retrieval function: returns the timeseries for a selected geometry ID.
3. **Public Assets for Dashboard:** Put styling and UI specification files (e.g., `form.json` and `chart.json`) into the mounted public bucket (`~/bucket/public/`) so they can be read by the dashboard.
4. **STAC Registration with custom `eodash` keys:** Register the collection on eoAPI, attaching the dynamic timeseries service link, and custom parameters `eodash:jsonform` and `eodash:vegadefinition` to drive interactive widgets in the frontend.

## 1. Setup & Environment
First, let's check our Python environment and import the required libraries.

In [ ]:
import sys
import os

# Auto-install any missing dependencies directly inside the Jupyter kernel
try:
    import psycopg
except ImportError:
    print("Installing psycopg...")
    %pip install -q psycopg

try:
    import pystac
except ImportError:
    print("Installing pystac...")
    %pip install -q pystac

try:
    import geopandas
except ImportError:
    print("Installing geopandas...")
    %pip install -q geopandas

import importlib.util

# Verify environment packages
required = ["pystac", "geopandas", "sqlalchemy", "pandas", "requests", "psycopg"]
missing = [p for p in required if importlib.util.find_spec(p) is None]

if missing:
    print(f"ERROR: Missing packages: {', '.join(missing)}")
    sys.exit("Execution stopped: Missing dependencies.")

import geopandas as gpd
import pandas as pd
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.exc import SAWarning
import warnings
from pystac import Collection, Extent, SpatialExtent, TemporalExtent, Link, Item
from datetime import datetime, timezone
import requests

# Suppress SQLAlchemy reflection warnings for unrecognized 'geometry' column types
warnings.filterwarnings("ignore", category=SAWarning)

print("Environment ready.")

## 2. Configuration
We use standard environment variables to connect to the `eoAPI` PostGIS database, define the STAC API URLs, and specify collection IDs.

In [ ]:
# Database connection settings (standard for eoAPI workspaces)
user = os.getenv("eoapi-db_user")
password = os.getenv("eoapi-db_password")
host = os.getenv("eoapi-db_host")
port = os.getenv("eoapi-db_port")
database = os.getenv("eoapi-db_dbname")

engine = create_engine(f"postgresql+psycopg://{user}:{password}@{host}:{port}/{database}")
inspector = inspect(engine)

# STAC & Vector Endpoint URLs
STAC_API_URL = "http://eoapi-rw-stac:8080"
eoapi_vector_endpoint = os.getenv("RASTER_ENDPOINT", "").replace("raster", "vector")

# Collection Settings
COLLECTION_ID = "mini_sample_collection"

# File Paths in the Repository
LOCAL_GEOM_PATH = "../../assets/mini_sample_geometries.geojson"
LOCAL_TS_PATH = "../../assets/mini_sample_timeseries.csv"

# Mounted public bucket directory path
# Any file copied to this folder can be accessed externally via the web
PUBLIC_DIR = f"{os.getenv('HOME')}/bucket/public/demo_sample_timeseries"
os.makedirs(PUBLIC_DIR, exist_ok=True)

## 3. Load Sample Datasets
Let's load our super-minimal sample dataset. It contains two sample regions and their respective timeseries data (stays, temperature, precipitation) over 3 days.

In [ ]:
# Load geometries using GeoPandas
gdf_geom = gpd.read_file(LOCAL_GEOM_PATH)
print("--- Geometries ---")
print(gdf_geom)

# Load timeseries using Pandas
df_ts = pd.read_csv(LOCAL_TS_PATH)
print("\n--- Timeseries Data ---")
print(df_ts)

## 4. Ingest into PostGIS (Separating Spatial & Temporal Data)
We write the spatial features to the geometry table and the timeseries records to the temporal table, keeping our database light and clean. We also create primary indices to maximize look-up and join performance.

In [ ]:
# 1. Ingest Unique Geometries (without using geoalchemy2)
# To avoid requiring geoalchemy2 (which is not in the kernel), we convert our shapes 
# to WKT (Well-Known Text), write them via standard pandas to_sql, and convert to geometry in SQL.
gdf_geom["geom_wkt"] = gdf_geom["geometry"].apply(lambda g: g.wkt)
df_geom_to_push = pd.DataFrame(gdf_geom.drop(columns=["geometry"]))

df_geom_to_push.to_sql(
    name="mini_sample_geometry",
    con=engine,
    if_exists="replace",
    index=False
)

# 2. Ingest Timeseries Data
df_ts.to_sql(
    name="mini_sample_timeseries",
    con=engine,
    if_exists="replace",
    index=False
)

# 3. Convert WKT column to proper geometry and create indices
with engine.begin() as conn:
    # Convert geom_wkt column to PostGIS geometry column
    conn.execute(text("""
        ALTER TABLE mini_sample_geometry ADD COLUMN geometry geometry(Geometry, 4326);
        UPDATE mini_sample_geometry SET geometry = ST_GeomFromText(geom_wkt, 4326);
        ALTER TABLE mini_sample_geometry DROP COLUMN geom_wkt;
    """))
    
    # Index on spatial table
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_mini_geom_id ON mini_sample_geometry(sample_id);"))
    # Indices on timeseries table
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_mini_ts_id ON mini_sample_timeseries(sample_id);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_mini_ts_time ON mini_sample_timeseries(time);"))
    conn.execute(text("CREATE INDEX IF NOT EXISTS idx_mini_ts_compound ON mini_sample_timeseries(sample_id, time);"))

print("✓ Data ingested & performance indices created successfully.")

## 5. Design PostGIS Functional Layers (Views with Parameters)
TiPg allows database functions to act as dynamic STAC/vector collection endpoints. We'll create two functions:
1. `mini_sample_for_date`: Accepts a `target_date` and dynamically joins timeseries with geometries for that day. This serves as our map overlay.
2. `timeseries_mini_sample_by_id`: Accepts a `target_id` and retrieves the timeseries profile for that feature, with a dummy geometry so TiPg can serve it as an item.

In [ ]:
# Execute PL/pgSQL function definitions
with engine.begin() as conn:
    # Dynamic Spatial Map Layer filtered by Date
    conn.execute(text("""
        CREATE OR REPLACE FUNCTION public.mini_sample_for_date(
            IN target_date DATE
        )
        RETURNS TABLE (
            sample_id TEXT,
            sample_region TEXT,
            region_type TEXT,
            "time" DATE,
            stays DOUBLE PRECISION,
            temp DOUBLE PRECISION,
            precip DOUBLE PRECISION,
            geom geometry(Geometry, 4326)
        )
        AS $$
        BEGIN
            RETURN QUERY
            SELECT 
                ts.sample_id,
                g.sample_region,
                g.region_type,
                CAST(ts.time AS DATE) AS "time",
                ts.stays,
                ts.temp,
                ts.precip,
                g.geometry AS geom
            FROM mini_sample_timeseries ts
            JOIN mini_sample_geometry g ON ts.sample_id = g.sample_id
            WHERE CAST(ts.time AS DATE) = target_date;
        END;
        $$ LANGUAGE plpgsql STABLE;
    """))

    # Timeseries Extractor filtered by Feature ID (includes ST_MakePoint for TiPg compatibility)
    conn.execute(text("""
        CREATE OR REPLACE FUNCTION public.timeseries_mini_sample_by_id(
            IN target_id TEXT
        )
        RETURNS TABLE (
            sample_id TEXT,
            "time" DATE,
            stays DOUBLE PRECISION,
            temp DOUBLE PRECISION,
            precip DOUBLE PRECISION,
            geom geometry(Geometry, 4326)
        )
        AS $$
        BEGIN
            RETURN QUERY
            SELECT 
                ts.sample_id,
                CAST(ts.time AS DATE) AS "time",
                ts.stays,
                ts.temp,
                ts.precip,
                ST_SetSRID(ST_MakePoint(0, 0), 4326)::geometry(Point, 4326) AS geom
            FROM mini_sample_timeseries ts
            WHERE ts.sample_id = target_id
            ORDER BY ts.time;
        END;
        $$ LANGUAGE plpgsql STABLE;
    """))

print("✓ Dynamic PostGIS view functions created successfully!")

## 6. Mounted Public Bucket & Dashboard Assets
We place the form definition (`mini_sample_form.json`) and the Vega-Lite chart definition (`mini_sample_chart.json`) into the mounted public directory (`~/bucket/public/demo_sample_timeseries/`). 

These files will be served at public URLs which we attach to our STAC Collection metadata so that the `eodash` client automatically knows how to render the interactive filter form and chart!

In [ ]:
import shutil

# Copy JSON assets from repo folder to the mounted public bucket directory
shutil.copy("../../assets/mini_sample_form.json", f"{PUBLIC_DIR}/form.json")
shutil.copy("../../assets/mini_sample_chart.json", f"{PUBLIC_DIR}/chart.json")
shutil.copy("../../assets/mini_sample_style.json", f"{PUBLIC_DIR}/style.json")

print("Files successfully copied to mounted public folder!")

# Resolve the public workspace base URL from the environment
workspace_public_url = os.getenv("workspaceconfig_publicurl")

if workspace_public_url:
    # Strip trailing slash if present to cleanly append paths
    public_base = workspace_public_url.rstrip("/")
    # In EOxHub workspaces, files in ~/bucket/public/ are served at: <workspaceconfig_publicurl>/<subpath>
    FORM_URL = f"{public_base}/demo_sample_timeseries/form.json"
    CHART_URL = f"{public_base}/demo_sample_timeseries/chart.json"
    STYLE_URL = f"{public_base}/demo_sample_timeseries/style.json"
else:
    # Fallback/placeholder URL if not running on the workspace server
    PUBLIC_BASE_URL = "https://workspace-ui-public.gtif-austria.hub-otc.eox.at/api/public/share/public-demo"
    FORM_URL = f"{PUBLIC_BASE_URL}/form.json"
    CHART_URL = f"{PUBLIC_BASE_URL}/chart.json"
    STYLE_URL = f"{PUBLIC_BASE_URL}/style.json"

print(f"Form URL: {FORM_URL}")
print(f"Chart URL: {CHART_URL}")
print(f"Style URL: {STYLE_URL}")

## 7. Create & Register STAC Collection with Dashboard Keys
Now we will create the STAC Collection. The crucial parts are:
1. **`eodash:jsonform`**: Points to the publicly shared JSON form.
2. **`eodash:vegadefinition`**: Points to the publicly shared Vega-Lite specification.
3. **Timeseries Service Link**: A `pystac.Link` with `rel="service"` and `id="timeseries"`. Its target HREF calls our PostGIS timeseries extraction endpoint, mapping the `target_id` dynamically with `{{feature.values_.sample_id }}` (interpreted on-click by `eodash`).

In [ ]:
# Define timeseries query service link (eodash parses sample_id from map selection)
ts_service_url = (
    f"{eoapi_vector_endpoint}/collections/public.timeseries_mini_sample_by_id/items?"
    "target_id={{feature.values_.sample_id }}&f=json&limit=10000&geom-column=#.json"
)

# Set temporal boundaries based on our dataset
start_time = datetime(2023, 1, 1, tzinfo=timezone.utc)
end_time = datetime(2023, 1, 3, tzinfo=timezone.utc)

# 1. Instantiate the Collection with custom eodash properties
coll = Collection(
    id=COLLECTION_ID,
    title="Minimal Sample and Timeseries Demo",
    description="Minimal demo showing timeseries separation, PostGIS parameterized functions, and eodash integration.",
    extent=Extent(
        SpatialExtent([[16.2, 47.5, 16.5, 47.9]]),
        TemporalExtent([[start_time, end_time]])
    ),
    license="other",
    extra_fields={
        "eodash:jsonform": FORM_URL,
        "eodash:vegadefinition": CHART_URL,
        "themes": ["Sample Demo"],
        "tags": ["sample", "timeseries", "vega"]
    }
)

# 2. Add timeseries retrieval service link
ts_link = Link(
    rel="service",
    target=ts_service_url,
    media_type="text/csv",
    title="Dynamic Timeseries Retrieval Service",
    extra_fields={
        "id": "timeseries",
        "method": "GET"
    }
)
coll.add_link(ts_link)

# 3. Register the Collection on STAC API
check_url = f"{STAC_API_URL}/collections/{COLLECTION_ID}"
resp = requests.get(check_url)
if resp.status_code == 200:
    print(f"Collection '{COLLECTION_ID}' already exists. Deleting first to reset...")
    requests.delete(check_url)

print(f"Registering collection '{COLLECTION_ID}' on eoAPI...")
post_resp = requests.post(f"{STAC_API_URL}/collections", json=coll.to_dict())

if post_resp.ok:
    print(f"✓ Success! Collection '{COLLECTION_ID}' successfully registered.")
else:
    print(f"ERROR: Failed to register collection. Status: {post_resp.status_code}")
    print(f"Details: {post_resp.text}")

## 8. Generate & Register STAC Items for Every Date Snapshot
We will now generate individual STAC Items for each unique date snapshot in our timeseries. 

**Crucial Details:**
1. **ID formatting:** Because the database (pgstac) does not permit colons (`:`) inside STAC Item IDs, we use the simple date representation (e.g., `2023-01-01`) instead of full ISO 8601 with colons.
2. **Dynamic Vector Tile Links:** We attach a `vector-tile` link pointing to our Dynamic PostGIS Date function, parameterizing it with the specific date query parameter: `?target_date=2023-01-01`.
3. **Style Link:** We attach a `style` link pointing to our shared stylesheet, matching our map rendering requirements.

In [ ]:
print("Generating and ingesting STAC Items for each unique date...")
unique_dates = sorted(df_ts["time"].unique())

# Compute general bounding box covering both regions
bbox = list(gdf_geom.total_bounds)  # [minx, miny, maxx, maxy]
geometry_dict = {
    "type": "Polygon",
    "coordinates": [[
        [bbox[0], bbox[1]],
        [bbox[2], bbox[1]],
        [bbox[2], bbox[3]],
        [bbox[0], bbox[3]],
        [bbox[0], bbox[1]]
    ]]
}

for dt_str in unique_dates:
    dt_val = datetime.strptime(dt_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    # Use the date string (no colons) as the ID to prevent pgstac validation errors
    item_id = dt_str
    
    # Create STAC Item representing this date's snapshot
    item = Item(
        id=item_id,
        geometry=geometry_dict,
        bbox=bbox,
        datetime=dt_val,
        properties={},
        collection=COLLECTION_ID
    )
    
    # Add the date-specific Vector Tile Link pointing to our PostGIS date function
    vt_url = f"{eoapi_vector_endpoint}/collections/public.mini_sample_for_date/tiles/WebMercatorQuad/{{z}}/{{x}}/{{y}}?target_date={dt_str}"
    vt_link = Link(
        rel="vector-tile",
        target=vt_url,
        media_type="application/vnd.mapbox-vector-tile",
        title="Minimal Sample & Timeseries Demo",
        extra_fields={
            "key": COLLECTION_ID
        }
    )
    item.add_link(vt_link)
    
    # Add the Style Link pointing to our publicly accessible style stylesheet
    style_link = Link(
        rel="style",
        target=STYLE_URL,
        media_type="text/vector-styles",
        extra_fields={
            "links:keys": [COLLECTION_ID]
        }
    )
    item.add_link(style_link)
    
    # Convert to dictionary and attach dummy asset as required by STAC/EOxHub
    item_dict = item.to_dict()
    item_dict["assets"] = {
        "dummy_asset": {
            "href": STYLE_URL
        }
    }
    
    # Post the STAC Item to eoAPI STAC endpoint
    items_url = f"{STAC_API_URL}/collections/{COLLECTION_ID}/items"
    
    # Delete first if it already exists to reset cleanly
    requests.delete(f"{items_url}/{item_id}")
    
    item_resp = requests.post(items_url, json=item_dict)
    if item_resp.ok:
        print(f"✓ Registered STAC Item for date: {dt_str} (ID: {item_id})")
    else:
        print(f"ERROR: Failed to register STAC Item for {dt_str}. Status: {item_resp.status_code}")
        print(f"Details: {item_resp.text}")